Phase 1 — Data Cleaning for ASD Screening Dataset
Reads the raw CSV, fixes known data-quality issues, and writes a clean
version to disk that Phase 2 (encoding) and Phase 3 (graph building)
will build on.

In [1]:
import pandas as pd
import numpy as np

In [2]:
RAW_PATH = "Data/autism_screening.csv"
CLEAN_PATH = "Data/data_clean.csv"

In [3]:
df = pd.read_csv(RAW_PATH)
print(f"Loaded raw data: {df.shape}")

Loaded raw data: (704, 21)


---------------------------------------------------------------
1. Drop zero-information / redundant columns
---------------------------------------------------------------
age_desc has a single constant value across all 704 rows -> no info.
result is essentially the sum of the 10 AQ items -> redundant with
the AQ item columns we're keeping individually.

In [4]:
df = df.drop(columns=["age_desc", "result"])

---------------------------------------------------------------
2. Fix the age column
---------------------------------------------------------------
One row has age=383 (clearly a data entry error). Treat it, and the
2 NaN rows, the same way: impute with the median age. We use the
median (not mean) because age is right-skewed and median is robust
to the very outlier we're trying to neutralize.

In [5]:
df.loc[df["age"] > 120, "age"] = np.nan  # treat impossible ages as missing
median_age = df["age"].median()
n_missing_age = df["age"].isnull().sum()
df["age"] = df["age"].fillna(median_age)
print(f"Age: imputed {n_missing_age} missing/invalid values with median={median_age}")

Age: imputed 3 missing/invalid values with median=27.0


---------------------------------------------------------------
3. Fix '?' placeholders in ethnicity and relation
---------------------------------------------------------------
'?' is not a real category, it's missingness. We recode it explicitly
as "Unknown" rather than silently one-hot-encoding '?' as if it were
a legitimate ethnicity/relation value.

In [6]:
df["ethnicity"] = df["ethnicity"].replace("?", "Unknown")
df["relation"] = df["relation"].replace("?", "Unknown")

Fix the ethnicity casing bug: 'Others' and 'others' are the same
category split by a typo in the original data collection.

In [7]:
df["ethnicity"] = df["ethnicity"].replace("others", "Others")

print(f"Ethnicity categories after cleaning: {sorted(df['ethnicity'].unique())}")
print(f"Relation categories after cleaning: {sorted(df['relation'].unique())}")

Ethnicity categories after cleaning: ['Asian', 'Black', 'Hispanic', 'Latino', 'Middle Eastern ', 'Others', 'Pasifika', 'South Asian', 'Turkish', 'Unknown', 'White-European']
Relation categories after cleaning: ['Health care professional', 'Others', 'Parent', 'Relative', 'Self', 'Unknown']


---------------------------------------------------------------
4. Bucket rare countries
---------------------------------------------------------------
contry_of_res has 67 unique values, many with only 1-2 patients.
One-hot encoding all 67 would blow up the feature space and dilute
the k-NN similarity computation in Phase 3 (curse of dimensionality:
with enough sparse dims, every patient looks equally "far" from
every other patient). We keep the top-N most frequent countries and
bucket the rest as "Other".

In [8]:
TOP_N_COUNTRIES = 10
top_countries = df["contry_of_res"].value_counts().nlargest(TOP_N_COUNTRIES).index
df["country_grouped"] = df["contry_of_res"].where(
    df["contry_of_res"].isin(top_countries), "Other"
)
print(f"\nCountry grouping -> kept top {TOP_N_COUNTRIES}, rest bucketed as 'Other':")
print(df["country_grouped"].value_counts())



Country grouping -> kept top 10, rest bucketed as 'Other':
country_grouped
Other                   154
United States           113
United Arab Emirates     82
New Zealand              81
India                    81
United Kingdom           77
Jordan                   47
Australia                27
Canada                   15
Sri Lanka                14
Afghanistan              13
Name: count, dtype: int64



drop the original high-cardinality country column, keep grouped version


In [9]:
df = df.drop(columns=["contry_of_res"])

---------------------------------------------------------------
5. Drop near-zero-variance column
---------------------------------------------------------------
used_app_before is "no" for 692/704 rows (98%). It carries almost no
discriminative signal and adds noise to the similarity computation.

In [10]:
print(f"\nused_app_before distribution:\n{df['used_app_before'].value_counts()}")
df = df.drop(columns=["used_app_before"])



used_app_before distribution:
used_app_before
no     692
yes     12
Name: count, dtype: int64


---------------------------------------------------------------
6. Normalize binary yes/no columns and target to clean 0/1 ints
   (kept as separate readable columns for now; full encoding happens
   in Phase 2)
---------------------------------------------------------------

In [11]:
df["jundice"] = df["jundice"].map({"no": 0, "yes": 1})
df["austim"] = df["austim"].map({"no": 0, "yes": 1})  # family history of autism
df["gender"] = df["gender"].map({"f": 0, "m": 1})
df["Class/ASD"] = df["Class/ASD"].map({"NO": 0, "YES": 1})

# rename austim -> family_autism_history for clarity (it's often confused
# with the target column at a glance)

In [12]:
df = df.rename(columns={"austim": "family_autism_history", "jundice": "jaundice"})

print(f"\nFinal cleaned shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass balance:\n{df['Class/ASD'].value_counts(normalize=True)}")

df.to_csv(CLEAN_PATH, index=False)
print(f"\nSaved cleaned data to {CLEAN_PATH}")


Final cleaned shape: (704, 18)
Columns: ['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 'age', 'gender', 'ethnicity', 'jaundice', 'family_autism_history', 'relation', 'Class/ASD', 'country_grouped']

Class balance:
Class/ASD
0    0.731534
1    0.268466
Name: proportion, dtype: float64

Saved cleaned data to Data/data_clean.csv
